In [ ]:
!pip install zarr
!pip install rasterio
from pathlib import Path
from typing import Iterable
from shapely.geometry import shape,Polygon
from math import ceil
import os, zarr, shapely, rasterio
import geopandas as gpd
import numpy as np
from rasterio.features import shapes
from shapely.ops import cascaded_union
from shapely.geometry import shape


#Change this to the folder containing your Xenium slide sub-folders
xenium_directory = '/content/drive/MyDrive/Acne Xenium/11629_JD_9_28_24'

def get_immediate_subdirectories(a_dir):
    return [name for name in os.listdir(a_dir)
            if os.path.isdir(os.path.join(a_dir, name))]

xenium_path_list = get_immediate_subdirectories(xenium_directory)

TOLERANCE_STEP = 0.5

class Versions:
    EXPERIMENT = [2, 0]
    GROUPS = [5, 0]
    CELL_CATEGORIES = [1, 0]

def cell_summary_attrs() -> dict:
    return {
        "column_descriptions": [
            "Cell centroid in X",
            "Cell centroid in Y",
            "Cell area",
            "Nucleus centroid in X",
            "Nucleus centroid in Y",
            "Nucleus area",
            "z_level",
        ],
        "column_names": [
            "cell_centroid_x",
            "cell_centroid_y",
            "cell_area",
            "nucleus_centroid_x",
            "nucleus_centroid_y",
            "nucleus_area",
            "z_level",
        ],
    }

def group_attrs() -> dict:
    return {
        "major_version": Versions.GROUPS[0],
        "minor_version": Versions.GROUPS[1],
        "name": "CellSegmentationDataset",
        "polygon_set_descriptions": [
            "NA",
            "NA",
        ],
        "polygon_set_display_names": ["Nucleus boundaries", "Cell boundaries"],
        "polygon_set_names": ["nucleus", "cell"],
        "spatial_units": "microns",
    }

def simplify_to_num_vertices(polygon, target_vertices, tolerance_step=0.1, max_iterations=100):
    """Simplifies a polygon to a target number of vertices."""

    current_tolerance = 0
    simplified_polygon = polygon

    for _ in range(max_iterations):
        simplified_polygon = polygon.simplify(current_tolerance)
        num_vertices = len(simplified_polygon.exterior.coords) - 1

        if num_vertices <= target_vertices:
            return simplified_polygon
        else:
            current_tolerance += tolerance_step

    return simplified_polygon


def pad_polygon(
    polygon: Polygon, max_vertices: int, tolerance: float = TOLERANCE_STEP
) -> np.ndarray:
    """Transform the polygon to have the desired number of vertices

    Args:
        polygon: A `shapely` polygon
        max_vertices: The desired number of vertices
        tolerance: The step of tolerance used for simplification. At each step, we increase the tolerance of this value until the polygon is simplified enough.

    Returns:
        A 2D array representing the polygon vertices
    """
    n_vertices = len(polygon.exterior.coords)
    assert n_vertices >= 3

    coords = polygon.exterior.coords._coords

    if n_vertices == max_vertices:
        return coords.flatten()

    if n_vertices < max_vertices:
        return np.pad(coords, ((0, max_vertices - n_vertices), (0, 0)), mode="edge").flatten()

    ''' if n_vertices > max_vertices:
        polygon = simplify_to_num_vertices(polygon = polygon, target_vertices = max_vertices) '''

    # TODO: improve it: how to choose the right tolerance?
    polygon = polygon.simplify(tolerance=tolerance)
    return pad_polygon(polygon, max_vertices, tolerance + TOLERANCE_STEP)
    #return polygon.exterior.coords._coords.flatten()


def write_polygons(
    path: Path,
    polygons: Iterable[Polygon],
    max_vertices: int,
    is_dir: bool = True,
    pixel_size: float = 0.2125,
) -> None:
    """Write a `cells.zarr.zip` file containing the cell polygonal boundaries

    Args:
        path: Path to the Xenium Explorer directory where the transcript file will be written
        polygons: A list of `shapely` polygons to be written
        max_vertices: The number of vertices per polygon (they will be transformed to have the right number of vertices)
        is_dir: If `False`, then `path` is a path to a single file, not to the Xenium Explorer directory.
        pixel_size: Number of microns in a pixel. Invalid value can lead to inconsistent scales in the Explorer.
    """

    print(f"Writing {len(polygons)} cell polygons")
    padded = []
    droplist = []
    for i, p in enumerate(polygons):
      #print(i)
      #print(p.exterior)
      try:
        padded.append(pad_polygon(p, max_vertices))
      except:
        droplist.append(i)
        continue

    print(droplist)
    for index in sorted(droplist, reverse=True):
      del polygons[index]

    coordinates = np.stack(padded)
    coordinates *= pixel_size

    num_cells = len(coordinates)
    cells_fourth = ceil(num_cells / 4)
    cells_half = ceil(num_cells / 2)

    GROUP_ATTRS = group_attrs()
    GROUP_ATTRS["number_cells"] = num_cells

    polygon_vertices = np.stack([coordinates, coordinates])
    num_points = polygon_vertices.shape[2]
    n_vertices = num_points // 2

    print(f'writing to {path}')
    if os.path. exists(os.path.join(path,'cells.zarr.zip')):
      os. remove(os.path.join(path,'cells.zarr.zip'))

    with zarr.ZipStore(os.path.join(path,'cells.zarr.zip'), mode="w") as store:
        g = zarr.group(store=store)
        g.attrs.put(GROUP_ATTRS)

        g.array(
            "polygon_vertices",
            polygon_vertices,
            dtype="float32",
            chunks=(1, cells_fourth, ceil(num_points / 4)),
        )

        cell_id = np.ones((num_cells, 2))
        cell_id[:, 0] = np.arange(num_cells)
        g.array("cell_id", cell_id, dtype="uint32", chunks=(cells_half, 1))

        cell_summary = np.zeros((num_cells, 7))
        cell_summary[:, 2] = [p.area for p in polygons]
        g.array(
            "cell_summary",
            cell_summary,
            dtype="float64",
            chunks=(num_cells, 1),
        )
        g["cell_summary"].attrs.put(cell_summary_attrs())

        g.array(
            "polygon_num_vertices",
            np.full((2, num_cells), n_vertices),
            dtype="int32",
            chunks=(1, cells_half),
        )

        g.array(
            "seg_mask_value",
            np.arange(num_cells),
            dtype="uint32",
            chunks=(cells_half,),
        )


In [ ]:

def retrieve_tif(path):
  with rasterio.open(path) as raster:
      image = raster.read(1).astype('float32')
      crs = 'EPSG:4326' #'raster.crs'
      list_pop = [
          {'cell_id': value, 'geometry': shape(shp)}
          for i, (shp, value)
          in enumerate(shapes(image, connectivity=4, transform=raster.transform))
          if value > 0
      ]
  df = gpd.GeoDataFrame(list_pop, crs=crs).to_crs(epsg=4326)
  return df

def fix_holes(df, geom = 'geometry'):
  for i, row in df.iterrows():
    if row[geom].geom_type == 'MultiPolygon':
        print(f'fixing holes in row {i}')
        M = row[geom]
        P = row[geom].geoms
        omega = shapely.concave_hull(M)
        print(omega)
        print(omega.area)
        df.geometry[i] = omega
  return df

" cell_df = retrieve_tif(input_raster)\ncell_df = cell_df.dissolve(by='cell_id', aggfunc='sum')\ncell_df['centroid'] = cell_df.geometry.centroid\ncell_df['x_centroid' ] = cell_df['centroid'].x\ncell_df['y_centroid' ] = cell_df['centroid'].y\ncell_df['cell_area'] = cell_df.geometry.area\ncell_df = fix_holes(cell_df)\ncell_df.head() "

In [ ]:
!pip uninstall spatialdata
!pip install spatialdata[extra]
import spatialdata_io
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
from shapely.validation import make_valid
from shapely import is_valid
import sys
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

#Cycle through Xenium folders and resegment
for xenium_path in xenium_path_list:
  print(xenium_path)
  xenium_path = os.path.join(xenium_directory, xenium_path)

  #Create sub-folder in Xenium slide folder for resegmented files
  resegment_path = os.path.join(xenium_path, 'resegmented')
  if os.path.exists(resegment_path):
    continue
  else:
    os.mkdir(resegment_path)
  print(resegment_path)

  #####IMPORT AND RESEGMENT KRT5+ CELLS
  xenium_data = spatialdata_io.xenium(xenium_path)

  gene_expression_table = xenium_data.get('table')

  #Pull in cell boundaries
  cell_df = xenium_data.get('cell_boundaries')
  cell_df['cell_area'] = cell_df.geometry.area
  cell_df['transcript_counts_start'] = gene_expression_table.obs['total_counts'].values
  cell_df = fix_holes(cell_df)
  cell_df['cell_boundaries'] = cell_df.geometry

  #Pull in nuclear boundaries
  nuclei_df = xenium_data.get('nucleus_boundaries')
  nuclei_df['nucleus_area'] = nuclei_df.geometry.area
  nuclei_df = fix_holes(nuclei_df)
  nuclei_df.geometry = nuclei_df.geometry.apply(make_valid)
  cell_df.geometry = cell_df.geometry.apply(make_valid)
  cell_nuclei_df = nuclei_df.sjoin(cell_df, how="inner", predicate='within')

  #Pull in transcripts
  column_names = list(xenium_data.get('transcripts').columns)
  column_dict = {}
  for i, col in enumerate(column_names):
    column_dict[i] = col
  print(column_dict)

  transcriptsDF = pd.DataFrame(xenium_data.get('transcripts'))
  transcriptsDF = transcriptsDF.rename(columns=column_dict)
  transcriptsDF = transcriptsDF[transcriptsDF['qv'] >= 20]
  transcriptsDF['geometry'] = gpd.points_from_xy(transcriptsDF.x, transcriptsDF.y)
  transcriptsDF.head()
  transcripts_gdf = gpd.GeoDataFrame(transcriptsDF, geometry="geometry")

  def get_genes_counts(df, gene1='KRT5', gene2='AWAT2'):
    K5AWAT_gdf = transcripts_gdf.sjoin(df, how="right")
    K5AWAT_gdf[gene1] = K5AWAT_gdf['feature_name'].str.contains(gene1, na=False)
    K5AWAT_gdf[gene2] = K5AWAT_gdf['feature_name'].str.contains(gene2, na=False)
    K5AWAT_gdf.index.name = 'idx'
    #Count KRT5 and AWAT2 for each cell
    krt5_count = K5AWAT_gdf.groupby(['cell_id'])[gene1].sum()
    awat2_count = K5AWAT_gdf.groupby(['cell_id'])[gene2].sum()

    return krt5_count, awat2_count

  #Define starting counts of KRT5 and AWAT2 in df
  krt5_count, awat2_count = get_genes_counts(cell_df)
  for i in range(len(krt5_count)):
    cell_df.loc[krt5_count.index[i], 'KRT5count_start'] = krt5_count.values[i]

  for i in range(len(awat2_count)):
    cell_df.loc[awat2_count.index[i], 'AWAT2count_start'] = awat2_count.values[i]

  krt79_count, fasn_count = get_genes_counts(cell_df, 'KRT79', 'FASN')
  for i in range(len(krt79_count)):
    cell_df.loc[krt79_count.index[i], 'KRT79count_start'] = krt79_count.values[i]

  for i in range(len(fasn_count)):
    cell_df.loc[fasn_count.index[i], 'FASNcount_start'] = fasn_count.values[i]

  pparg_count, igfbp4_count = get_genes_counts(cell_df, 'PPARG', 'IGFBP4')
  for i in range(len(pparg_count)):
    cell_df.loc[pparg_count.index[i], 'PPARGcount_start'] = pparg_count.values[i]

  for i in range(len(igfbp4_count)):
    cell_df.loc[igfbp4_count.index[i], 'IGFBP4count_start'] = igfbp4_count.values[i]

  #Set initial buffering boolean
  for i, row in cell_df.iterrows():
    if row['KRT5count_start'] >= 3:
      cell_df.loc[i, 'Buffering'] = True
    else:
      cell_df.loc[i, 'Buffering'] = False

  print(cell_df['Buffering'].value_counts())

  cell_transcripts_df = transcripts_gdf.sjoin(cell_df, how="inner")
  K5_gdf = cell_transcripts_df[cell_transcripts_df['feature_name'].str.startswith('KRT5')]

  cell_df['geometry'] = cell_df['cell_boundaries']
  cell_df = cell_df[~cell_df['transcript_counts_start'].isnull()]
  cell_df = fix_holes(cell_df)

  for i, row in cell_df.iterrows():
    if row['Buffering'] == True:
      shapes_list = []
      cell_transcripts = K5_gdf[K5_gdf['index_right'] == i]
      cell_nuclei = cell_nuclei_df[cell_nuclei_df['index_right'] == i]
      Transcripts_union = shapely.convex_hull(cell_transcripts.geometry.unary_union)
      Nuclei_union = cell_nuclei.geometry.unary_union
      try:
        Shape_union = shapely.unary_union([Transcripts_union, Nuclei_union])
      except:
        continue
      if type(Shape_union) != shapely.geometry.polygon.Polygon:
        continue
      cell_df.loc[i, 'geometry'] = Shape_union

  cell_df = fix_holes(cell_df)
  #print(cell_df)

  #####Build cells.zarr.zip file
  krt5_count, awat2_count = get_genes_counts(cell_df)
  for i in range(len(krt5_count)):
    cell_df.loc[krt5_count.index[i], 'KRT5count_finish'] = krt5_count.values[i]

  for i in range(len(awat2_count)):
    cell_df.loc[awat2_count.index[i], 'AWAT2count_finish'] = awat2_count.values[i]

  krt79_count, fasn_count = get_genes_counts(cell_df, 'KRT79', 'FASN')
  for i in range(len(krt79_count)):
    cell_df.loc[krt79_count.index[i], 'KRT79count_finish'] = krt79_count.values[i]

  for i in range(len(fasn_count)):
    cell_df.loc[fasn_count.index[i], 'FASNcount_finish'] = fasn_count.values[i]

  pparg_count, igfbp4_count = get_genes_counts(cell_df, 'PPARG', 'IGFBP4')
  for i in range(len(pparg_count)):
    cell_df.loc[pparg_count.index[i], 'PPARGcount_finish'] = pparg_count.values[i]

  for i in range(len(igfbp4_count)):
    cell_df.loc[igfbp4_count.index[i], 'IGFBP4count_finish'] = igfbp4_count.values[i]

  cell_df = cell_df[~cell_df['transcript_counts_start'].isnull()]



  #Build cell feature matrix
  #Rebuild cell_transcripts_df for new cell_df
  cell_transcripts_df = transcripts_gdf.sjoin(cell_df, how="inner")
  print(cell_transcripts_df)

  cell_transcripts_df = cell_transcripts_df[~cell_transcripts_df['feature_name'].str.startswith('Neg')]
  cell_transcripts_df = cell_transcripts_df[~cell_transcripts_df['feature_name'].str.startswith('Unassigned')]
  cell_feature_matrix = pd.DataFrame(index = cell_df.index, columns = cell_transcripts_df['feature_name'].unique())
  cell_feature_matrix.fillna(0, inplace=True)

  print(cell_feature_matrix)

  for i, row in cell_transcripts_df.iterrows():
    cell_feature_matrix.loc[row['index_right'], row['feature_name']] += 1

  print(cell_feature_matrix)

  cell_feature_path = os.path.join(resegment_path, 'cell_feature_matrix.csv')
  print(f'printing to {cell_feature_path}')
  cell_feature_matrix.to_csv(cell_feature_path)

  #Build cells.czv file
  #cell_df['cell_id'] = cell_df.index
  #cell_df.drop(columns='cell_id', inplace = True)
  cell_df['centroid'] = cell_df.geometry.centroid
  cell_df['x_centroid'] = cell_df['centroid'].x
  cell_df['y_centroid'] = cell_df['centroid'].y
  cell_df['cell_area'] = cell_df.geometry.area
  cell_df['transcript_counts'] = cell_df.index.map(cell_feature_matrix.sum(axis = 1))
  cell_df['control_probe_counts'] = 0
  cell_df['control_codeword_counts'] = 0
  cell_df['unassigned_codeword_counts'] = 0
  cell_df['deprecated_codeword_counts'] = 0
  cell_df['total_counts'] = cell_df['transcript_counts'] + cell_df['control_probe_counts'] + cell_df['control_codeword_counts'] + cell_df['unassigned_codeword_counts'] + cell_df['deprecated_codeword_counts']

  cell_df_pd = pd.DataFrame(cell_df)
  cell_df_pd.reset_index(inplace=True)
  cell_df_pd.drop(columns=['geometry', 'centroid'], inplace=True)
  print(cell_df_pd)
  cell_df_pd.to_csv(os.path.join(resegment_path, 'cells.csv.gz'), index=False)

P7_NL_healthy_face_output-XETG00077__0040330__11629-JD-2_ROI_D1__20240911__170105
P3_L_comedone_back_output-XETG00077__0040330__11629-JD-2_ROI_C2__20240911__170105
P3_L_back_comedone_output-XETG00077__0040330__11629-JD-2_ROI_C1__20240911__170105
P1_L_face_pustule_output-XETG00077__0040378__11629-JD-1_ROI_B3__20240911__170105
P3_L_face_output-XETG00077__0040330__11629-JD-2_ROI_B1__20240911__170105
P1_NL_output-XETG00077__0040378__11629-JD-1_ROI_A1__20240911__170105
P6_NL_output-XETG00077__0040330__11629-JD-2_ROI_A1__20240911__170105
P2_NL_output-XETG00077__0040378__11629-JD-1_ROI_B1__20240911__170105
P6_L_face_comedo_output-XETG00077__0040378__11629-JD-1_ROI_C1__20240911__170105
/content/drive/MyDrive/Acne Xenium/11629_JD_9_28_24/P6_L_face_comedo_output-XETG00077__0040378__11629-JD-1_ROI_C1__20240911__170105/resegment_2KRT_no_buffer
INFO     reading /content/drive/MyDrive/Acne                                                                       
         Xenium/11629_JD_9_28_24/P6_L_fa

<ipython-input-9-3e7204b95169>:165: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cell_feature_matrix.fillna(0, inplace=True)


            TP53  VIM  SGPL1  IGFBP4  EGFR  ACACA  COL1A1  IGF1R  ZBTB16  \
aaaidcic-1     0    0      0       0     0      0       0      0       0   
aaaijegk-1     0    0      0       0     0      0       0      0       0   
aaaijeop-1     0    0      0       0     0      0       0      0       0   
aaalflje-1     0    0      0       0     0      0       0      0       0   
aaamdebo-1     0    0      0       0     0      0       0      0       0   
...          ...  ...    ...     ...   ...    ...     ...    ...     ...   
ohkaggha-1     0    0      0       0     0      0       0      0       0   
ohkfhjlj-1     0    0      0       0     0      0       0      0       0   
ohkhdbci-1     0    0      0       0     0      0       0      0       0   
ohkkefkn-1     0    0      0       0     0      0       0      0       0   
ohkooipd-1     0    0      0       0     0      0       0      0       0   

            PECAM1  ...  CD8A  NRG4  IL17A  NLRP3  MMP3  CD3D  MMP1  CYP19A1  \
aaaidci